In [1]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.2 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires nu

In [2]:
from ultralytics import YOLO

# Kiểm tra xem YOLO có hoạt động và nhận diện đúng GPU không
model = YOLO('yolov8n.pt')
print("YOLOv8 đã sẵn sàng! Thiết bị hỗ trợ:", model.device)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOv8 đã sẵn sàng! Thiết bị hỗ trợ: cpu


In [3]:
import os
import random
from tqdm.notebook import tqdm

# 1. Đường dẫn gốc từ Input Kaggle của bạn
src_img_dir = '/kaggle/input/datasets/lylmsc/wider-face-for-yolo-training/images'
src_lbl_dir = '/kaggle/input/datasets/lylmsc/wider-face-for-yolo-training/labels'

# 2. Thư mục đích trong vùng có quyền ghi
base_output_dir = '/kaggle/working/wider_face_split'

# Cấu trúc gồm cả train, val và test
split_dirs = [
    'images/train', 'images/val', 'images/test',
    'labels/train', 'labels/val', 'labels/test'
]

# Tạo các thư mục đích nếu chưa tồn tại
for s_dir in split_dirs:
    os.makedirs(os.path.join(base_output_dir, s_dir), exist_ok=True)

# 3. Lấy và trộn danh sách ảnh
all_images = [f for f in os.listdir(src_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
random.seed(42)
random.shuffle(all_images)

# 4. Tính toán mốc chia tỉ lệ (80% Train, 10% Val, 10% Test)
total_count = len(all_images)
train_end = int(total_count * 0.8)
val_end = train_end + int(total_count * 0.1)

train_images = all_images[:train_end]
val_images = all_images[train_end:val_end]
test_images = all_images[val_end:]

def link_files_with_progress(image_list, split_type):
    """Hàm tạo liên kết ảo có hiển thị thanh tiến độ"""
    # Sử dụng tqdm để vẽ thanh tiến trình trên Kaggle Notebook
    for img_name in tqdm(image_list, desc=f"Đang xử lý tập {split_type.upper()}"):
        base_name = os.path.splitext(img_name)[0]
        lbl_name = base_name + '.txt'
        
        src_img = os.path.join(src_img_dir, img_name)
        src_lbl = os.path.join(src_lbl_dir, lbl_name)
        
        dst_img = os.path.join(base_output_dir, 'images', split_type, img_name)
        dst_lbl = os.path.join(base_output_dir, 'labels', split_type, lbl_name)
        
        if os.path.exists(src_img) and not os.path.exists(dst_img):
            os.symlink(src_img, dst_img)
            
        if os.path.exists(src_lbl) and not os.path.exists(dst_lbl):
            os.symlink(src_lbl, dst_lbl)

# Tiến hành chạy và hiển thị tiến độ của từng tập dữ liệu
print(f"Tổng số lượng ảnh tìm thấy: {total_count} ảnh.\n")
link_files_with_progress(train_images, 'train')
link_files_with_progress(val_images, 'val')
link_files_with_progress(test_images, 'test')

print("\n--- PHÂN CHIA HOÀN THÀNH ---")
print(f"Tập Train : {len(train_images)} ảnh")
print(f"Tập Val   : {len(val_images)} ảnh")
print(f"Tập Test  : {len(test_images)} ảnh")

Tổng số lượng ảnh tìm thấy: 12880 ảnh.



Đang xử lý tập TRAIN:   0%|          | 0/10304 [00:00<?, ?it/s]

Đang xử lý tập VAL:   0%|          | 0/1288 [00:00<?, ?it/s]

Đang xử lý tập TEST:   0%|          | 0/1288 [00:00<?, ?it/s]


--- PHÂN CHIA HOÀN THÀNH ---
Tập Train : 10304 ảnh
Tập Val   : 1288 ảnh
Tập Test  : 1288 ảnh


In [4]:
import yaml

data_config = {
    'path': '/kaggle/working/wider_face_split',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',  # Bổ sung tập test vào cấu hình
    
    'names': {
        0: 'face'
    }
}

with open('/kaggle/working/dataset_split.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("Đã cập nhật file cấu hình dataset_split.yaml thành công!")

Đã cập nhật file cấu hình dataset_split.yaml thành công!


In [ ]:
from ultralytics import YOLO
import torch

# 1. Kiểm tra chắc chắn xem Kaggle Notebook đã nhận diện được GPU chưa
if torch.cuda.is_available():
    print(f"✅ Đã tìm thấy GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ CẢNH BÁO: Chưa bật GPU! Hãy vào mục 'Accelerator' ở góc phải màn hình Kaggle và chọn GPU T4 x2 hoặc P100.")

# 2. Khởi tạo mô hình
model = YOLO('yolov8n.pt') 

# 3. Tiến hành huấn luyện trên GPU
results = model.train(
    # Cấu hình file và tài nguyên
    data='/kaggle/working/dataset_split.yaml',
    epochs=50,       
    imgsz=640,       
    batch=16,        
    
    # --- CẤU HÌNH GPU ---
    device=0,             # Ép buộc sử dụng GPU số 0 (CUDA:0). Nếu muốn dùng CPU thì để 'cpu'
    workers=2,            # Số luồng CPU hỗ trợ nạp data vào GPU (đối với Kaggle nên để 2 hoặc 4)
    
    # --- CẤU HÌNH NGƯỠNG CONF & IOU ---
    conf=0.25,            # Ngưỡng tin cậy tối thiểu để tính toán loss/accuracy trong khi train
    iou=0.45,             # Ngưỡng giao thoa khi xử lý các hộp trùng nhau
    
    # --- CÁC THÔNG SỐ TỐI ƯU KHÁC ---
    optimizer='AdamW',    
    lr0=0.001,           
    cos_lr=True,         
    close_mosaic=10,     
    save=True,           
    val=True,             
    cache=False           # Để False để tránh tràn bộ nhớ RAM của Kaggle
)

✅ Đã tìm thấy GPU: Tesla T4
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=0.25, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset_split.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.45, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, 